# GPT-1 (openai-gpt) Dual Fine-tuning Experiment

이 노트북은 다음 실험을 수행합니다.

1. `GPT_A`: 감성 분류 데이터로 파인튜닝
2. `GPT_B`: QA 데이터로 파인튜닝
3. 두 모델 모두에 대해 감성/QA 입력을 넣고 결과를 비교

실행 환경: Google Colab + A100 GPU + High-RAM

In [ ]:
# Colab 환경에서 최초 1회 실행
!pip -q install -U transformers datasets accelerate evaluate scikit-learn pandas

In [ ]:
import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_MODEL = "openai-gpt"  # GPT-1
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("pad token:", tokenizer.pad_token)

In [ ]:
# -----------------------------
# 1) Sentiment dataset (SST-2)
# -----------------------------
sst2 = load_dataset("glue", "sst2")

# Colab 실험 시간 단축을 위해 샘플링 (필요 시 숫자 증가)
SENT_TRAIN_N = 8000
SENT_VAL_N = 1000

sst2_train = sst2["train"].shuffle(seed=SEED).select(range(SENT_TRAIN_N))
sst2_val = sst2["validation"].shuffle(seed=SEED).select(range(SENT_VAL_N))

def preprocess_sst2(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

sst2_train_tok = sst2_train.map(preprocess_sst2, batched=True)
sst2_val_tok = sst2_val.map(preprocess_sst2, batched=True)

sst2_train_tok = sst2_train_tok.rename_column("label", "labels")
sst2_val_tok = sst2_val_tok.rename_column("label", "labels")

sst2_train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
sst2_val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(sst2_train_tok)
print(sst2_val_tok)

# -----------------------------
# 2) QA dataset (SQuAD v1)
# -----------------------------
squad = load_dataset("squad")

QA_TRAIN_N = 20000
QA_VAL_N = 2000

qa_train = squad["train"].shuffle(seed=SEED).select(range(QA_TRAIN_N))
qa_val = squad["validation"].shuffle(seed=SEED).select(range(QA_VAL_N))


def preprocess_qa(batch):
    tokenized = tokenizer(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        padding="max_length",
        return_offsets_mapping=True,
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        answer = batch["answers"][i]
        answer_start = answer["answer_start"][0]
        answer_text = answer["text"][0]
        answer_end = answer_start + len(answer_text)

        sequence_ids = tokenized.sequence_ids(i)

        context_start = 0
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1

        if context_start >= len(sequence_ids) or context_end < 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        if offsets[context_start][0] > answer_start or offsets[context_end][1] < answer_end:
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_idx = context_start
            while start_idx <= context_end and offsets[start_idx][0] <= answer_start:
                start_idx += 1
            start_positions.append(start_idx - 1)

            end_idx = context_end
            while end_idx >= context_start and offsets[end_idx][1] >= answer_end:
                end_idx -= 1
            end_positions.append(end_idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized.pop("offset_mapping")

    return tokenized

qa_train_tok = qa_train.map(preprocess_qa, batched=True, remove_columns=qa_train.column_names)
qa_val_tok = qa_val.map(preprocess_qa, batched=True, remove_columns=qa_val.column_names)

qa_train_tok.set_format(type="torch")
qa_val_tok.set_format(type="torch")

print(qa_train_tok)
print(qa_val_tok)

In [ ]:
# -----------------------------
# 3) Fine-tune GPT_A (Sentiment)
# -----------------------------
gpt_a = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
gpt_a.config.pad_token_id = tokenizer.pad_token_id

args_a = TrainingArguments(
    output_dir="./gpt_a_sentiment",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_a = Trainer(
    model=gpt_a,
    args=args_a,
    train_dataset=sst2_train_tok,
    eval_dataset=sst2_val_tok,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

trainer_a.train()
print(trainer_a.evaluate())

trainer_a.save_model("./GPT_A")
tokenizer.save_pretrained("./GPT_A")
print("Saved GPT_A at ./GPT_A")

In [ ]:
# -----------------------------
# 4) Fine-tune GPT_B (Question Answering)
# -----------------------------
gpt_b = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL)
gpt_b.config.pad_token_id = tokenizer.pad_token_id

args_b = TrainingArguments(
    output_dir="./gpt_b_qa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_b = Trainer(
    model=gpt_b,
    args=args_b,
    train_dataset=qa_train_tok,
    eval_dataset=qa_val_tok,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

trainer_b.train()
print(trainer_b.evaluate())

trainer_b.save_model("./GPT_B")
tokenizer.save_pretrained("./GPT_B")
print("Saved GPT_B at ./GPT_B")

In [ ]:
import torch.nn.functional as F

id2label = {0: "negative", 1: "positive"}


def predict_sentiment(model, text):
    model.eval()
    model.to(device)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()
    pred = int(np.argmax(probs))
    return {
        "text": text,
        "pred_label": id2label.get(pred, str(pred)),
        "probs": probs.tolist(),
    }


def answer_question(model, question, context):
    model.eval()
    model.to(device)
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LEN,
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits, dim=-1).item()
    end = torch.argmax(outputs.end_logits, dim=-1).item()
    if end < start:
        end = start

    answer_ids = inputs["input_ids"][0][start : end + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return {
        "question": question,
        "answer": answer,
    }

# 실제 예시 입력
sentiment_examples = [
    "This movie is absolutely wonderful and touching.",
    "The product quality is terrible and I want a refund.",
]

qa_question = "Where do I live?"
qa_context = "My name is Minji, and I live in Seoul with my family."

print("=" * 80)
print("[GPT_A] sentiment fine-tuned model")
print("=" * 80)
for text in sentiment_examples:
    print("Sentiment input:", text)
    print("Sentiment output:", predict_sentiment(gpt_a, text))
    print()

print("QA input:")
print("Q:", qa_question)
print("C:", qa_context)
try:
    qa_result_a = answer_question(
        AutoModelForQuestionAnswering.from_pretrained("./GPT_A").to(device),
        qa_question,
        qa_context,
    )
    print("QA output:", qa_result_a)
except Exception as e:
    print("QA output: GPT_A is sentiment head, so direct QA head loading can fail.")
    print("Error:", str(e)[:300])

print("\n" + "=" * 80)
print("[GPT_B] QA fine-tuned model")
print("=" * 80)
print("QA input:")
print("Q:", qa_question)
print("C:", qa_context)
print("QA output:", answer_question(gpt_b, qa_question, qa_context))

print()
for text in sentiment_examples:
    print("Sentiment input:", text)
    try:
        sent_result_b = predict_sentiment(
            AutoModelForSequenceClassification.from_pretrained("./GPT_B", num_labels=2).to(device),
            text,
        )
        print("Sentiment output:", sent_result_b)
    except Exception as e:
        print("Sentiment output: GPT_B is QA head, so direct sentiment head loading can fail.")
        print("Error:", str(e)[:300])
    print()

## 실행 가이드

1. 위에서부터 순서대로 실행
2. A100 + High-RAM 기준으로도 학습 시간이 걸릴 수 있음
3. 더 빠르게 보려면 `SENT_TRAIN_N`, `QA_TRAIN_N`, `num_train_epochs`를 줄이기
4. 더 정확하게 보려면 데이터 수와 epoch를 늘리기

참고:
- `GPT_A`는 감성 분류 head, `GPT_B`는 QA head라서 서로 다른 태스크를 직접 수행할 때 오류 또는 품질 저하가 날 수 있음
- 이 노트북은 "한 태스크로 파인튜닝한 모델이 다른 태스크에서 어떻게 보이는지"를 실험적으로 확인하기 위한 구성입니다.